In [2]:
import torch 

torch.cuda.is_available()

True

In [3]:
torch.__version__

'2.5.1+cu121'

In [4]:
t0 =torch.tensor(1)
t1d = torch.tensor([1,2,3])
t2d = torch.tensor([[1,2],
                    [3, 4]])
t3d = torch.tensor([[[1,2],[3,4]],
                    [[5,6],[7,8]]])

print(t0)
print(t1d)
print(t2d)
print(t3d)

print(t1d.dtype)

tensor(1)
tensor([1, 2, 3])
tensor([[1, 2],
        [3, 4]])
tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
torch.int64


In [5]:
floadvec = torch.tensor([1.0, 2.0, 3.0])
print(floadvec.dtype)

torch.float32


In [19]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

grad_L1_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

In [20]:
print(grad_L1_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [21]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


# Neural Network Model

In [22]:
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            torch.nn.Linear(20, num_outputs)
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [33]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model)
print(model.layers[0].weight)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [24]:
num_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total numer of trainable model parameters:", num_param)

Total numer of trainable model parameters: 2213


In [27]:
print(model.layers[0].weight.shape)
print(model.layers[0].bias.shape)

torch.Size([30, 50])
torch.Size([30])


In [35]:
torch.manual_seed(123)
x = torch.rand([1, 50])
with torch.no_grad():
    out = model(x)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


# Data Loaders

In [38]:
x_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0 , 0, 0, 1, 1])

x_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])
y_test = torch.tensor([0, 1])

In [40]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, x, y):
        self.features = x
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    
    def __len__(self):
        return self.labels.shape[0]
    

train_ds = ToyDataset(x_train, y_train)
test_ds = ToyDataset(x_test, y_test)

In [43]:
print(train_ds[2])

(tensor([-0.5000,  2.6000]), tensor(0))


In [50]:
from torch.utils.data import DataLoader
    
torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

In [51]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])


# Training Loop

In [68]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.5
)

num_epochs = 3
for epoch in range(num_epochs):
    
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits =  model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d} | Batch: {batch_idx: 03d}/{len(train_loader):03d} | Train Loss: {loss:.2f}")
    
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
predictions = torch.argmax(probas, dim=1)
print(probas)
print(predictions)
print(predictions == y_train)
print(torch.sum(predictions == y_train))
 


Epoch: 001/003 | Batch:  00/002 | Train Loss: 0.75
Epoch: 001/003 | Batch:  01/002 | Train Loss: 0.65
Epoch: 002/003 | Batch:  00/002 | Train Loss: 0.44
Epoch: 002/003 | Batch:  01/002 | Train Loss: 0.13
Epoch: 003/003 | Batch:  00/002 | Train Loss: 0.03
Epoch: 003/003 | Batch:  01/002 | Train Loss: 0.00
tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])
tensor([[    0.9991,     0.0009],
        [    0.9982,     0.0018],
        [    0.9949,     0.0051],
        [    0.0491,     0.9509],
        [    0.0307,     0.9693]])
tensor([0, 0, 0, 1, 1])
tensor([True, True, True, True, True])
tensor(5)


In [59]:
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad == True))

Trainable parameters: 752


In [72]:
def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):
        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = predictions == labels
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item() 

In [73]:
compute_accuracy(model, train_loader)

1.0

In [74]:
compute_accuracy(model, test_loader)

1.0

# Saving and loading models

In [75]:
torch.save(model.state_dict(), "model.pth")

In [77]:
recover_model = NeuralNetwork(2, 2)
recover_model.load_state_dict(torch.load("model.pth"))

/tmp/ipykernel_2975250/3118076010.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  recover_model.load_state_dict(torch.load("model.pth"))


<All keys matched successfully>

In [79]:
model.state_dict().keys()

odict_keys(['layers.0.weight', 'layers.0.bias', 'layers.2.weight', 'layers.2.bias', 'layers.4.weight', 'layers.4.bias'])

# Training on GPU

In [80]:
torch.cuda.is_available()

True

In [86]:
t1 = torch.tensor([1., 2., 3.])
t2 = torch.tensor([4., 5., 6.])

print(t1 + t2)

tensor([5., 7., 9.])


In [87]:
t1 = t1.to("cuda")
t2 = t2.to("cuda")

print(t1+t2)

tensor([5., 7., 9.], device='cuda:0')


In [90]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

device = torch.device("cuda")
modle = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
        
    model.eval()

Epoch 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch 003/003 | Batch 001/002 | Train/Val Loss: 0.00
